# Cookbook: Web IQ — Grounding Agents with Real-Time Web Search

AI applications are only as good as the information they reason from. Some questions can only be answered with information that changes by the hour — today's news, a product launched this morning, the current weather, the latest version of a doc page. **Microsoft Web IQ** closes that gap.

Web IQ is Microsoft's agent-native grounding service: a suite of AI-native APIs that return **ranked, citation-ready context** from across the web — web pages, news, images, and videos — designed for direct injection into an LLM's context window. It's built on 20+ years of Bing search infrastructure and re-architected for LLMs and multi-step agents:

| | |
|---|---|
| ⚡ **Speed** | 164 ms P95 latency — built to keep multi-step agent chains moving |
| 🎯 **Quality** | Ranked for relevance, freshness, and authority — validated against grounding benchmarks such as DeepSearchQA |
| 🪙 **Efficiency** | Passage-level retrieval surfaces only the most relevant text, so you send fewer tokens per query |
| 📚 **Content** | Combines licensed sources, structured data, and the open web — not SERP scraping |
| 🔌 **Open** | REST, MCP (JSON-RPC 2.0), and SDKs — model-agnostic, no inference lock-in |

### What you'll build

By the end of this cookbook you'll have called every Web IQ API, compared how much context each content format sends to your model, grounded a model's answer in live sources with numbered citations, and connected Web IQ to VS Code as an MCP server.

| Step | Topic | API |
|---|---|---|
| 1 | Web Search and the response anatomy | `POST /search/web` |
| 2 | Content formats and token efficiency | `POST /search/web` |
| 3 | Targeting: `site:`, language, region, location | `POST /search/web` |
| 4 | News Search (last 14 days) | `POST /search/news` |
| 5 | Images Search with filters | `POST /search/images` |
| 6 | Videos Search with moments | `POST /search/videos` |
| 7 | Browse a specific URL (with live crawl) | `POST /browse` |
| 8 | Auto (Beta) — typed evidence with citations | `POST /auto` |
| 9 | Ground a model's answer with citations | Web IQ + Microsoft Foundry *(optional)* |
| 10 | Errors, rate limits, and retries | All |
| 11 | Web IQ as an MCP server | `https://api.microsoft.ai/v3/mcp` |
| 12 | Instrumentation: citation and click telemetry | Ping URLs |

## Prerequisites

1. **Web IQ access** — Web IQ is currently in **limited access** for select enterprise customers building AI agents and applications at scale. Request access at [webiq.microsoft.ai](https://webiq.microsoft.ai/).
   > 💡 **Can't verify your invitation code?** The invitation is usually tied to a different address than the one you sign in with. Open [portal.azure.com](https://portal.azure.com), check the address shown under your name in the top-right corner, and reply to your invitation email with that address.
2. **An API key** — In the [Web IQ Portal → Profile Management](https://webiq.microsoft.ai/profiles/), open the **Authentication** tab and select **Create API Key**. The same page shows your **quota** (for example, `60 QPM`) and which **services** (Web, Videos, Browse, News, Images) your profile is allowed to call.
3. **Python 3.10+**.
4. *(Optional, Step 9 only)* A **Microsoft Foundry project** with a chat model deployment, and `az login` so `DefaultAzureCredential` can authenticate. Every other step needs only the Web IQ API key.

> 🧪 **Tip:** The [Web IQ Playground](https://webiq.microsoft.ai/playground/) lets you build any request interactively and copy the generated code in cURL, Python, JavaScript, C#, Java, Go, or Ruby.

## Setup

Copy `.env.sample` to `.env` in this folder and fill in your values:

```
WEBIQ_API_KEY=<your-web-iq-api-key>
WEBIQ_BASE_URL=https://api.microsoft.ai/v3

# Optional — only for Step 9
FOUNDRY_PROJECT_ENDPOINT=https://<your-foundry-resource>.services.ai.azure.com/api/projects/<your-project>
FOUNDRY_MODEL_DEPLOYMENT_NAME=gpt-4o-mini
```

In [ ]:
%%capture
%pip install -U requests python-dotenv azure-identity "azure-ai-projects>=2.0.0"

### Authenticate

Every Web IQ request is an HTTPS `POST` with a JSON body and one of two credentials:

| Mode | Header | When to use |
|---|---|---|
| **API key** *(this cookbook)* | `x-apikey: <key>` | Quick start, prototypes, server-side apps |
| **Entra ID OAuth 2.0** (client credentials) | `Authorization: Bearer <token>` | Production apps that already use an Entra ID app registration |

To use Entra ID: in **Profile Management → Authentication → Application (Client) IDs**, select **Bind Application (Client) ID** and enter your app registration's client ID (allow ~1 minute to sync). Then request an app-only token with scope `https://api.microsoft.ai/.default`, for example with `azure.identity.ClientSecretCredential(tenant_id, client_id, client_secret).get_token("https://api.microsoft.ai/.default")`.

> ⚠️ Web IQ requires **HTTPS** (plain HTTP returns `410 Gone`) and a `content-type: application/json` header (otherwise `415`).

In [ ]:
import os
import json
import time
import urllib.parse

import requests
from dotenv import load_dotenv

load_dotenv(override=True)

WEBIQ_API_KEY = os.environ["WEBIQ_API_KEY"]
WEBIQ_BASE_URL = os.environ.get("WEBIQ_BASE_URL", "https://api.microsoft.ai/v3").rstrip("/")

session = requests.Session()
session.headers.update({
    "x-apikey": WEBIQ_API_KEY,
    "content-type": "application/json",
})


class WebIQError(Exception):
    def __init__(self, status: int, body: dict):
        self.status = status
        self.body = body
        super().__init__(
            f"{status} {body.get('errorCode')}: {body.get('userMessage')} "
            f"(traceId={body.get('traceId')})"
        )


RETRYABLE = {429, 500, 503, 504}


def webiq_post(path: str, payload: dict, max_retries: int = 4) -> dict:
    # Calls a Web IQ REST endpoint, retrying transient failures (see Step 10). Raises WebIQError otherwise.
    for attempt in range(max_retries + 1):
        response = session.post(f"{WEBIQ_BASE_URL}{path}", json=payload, timeout=60)
        if response.status_code == 200:
            return response.json()
        try:
            body = response.json()
        except ValueError:
            body = {"userMessage": response.text}
        retryable = response.status_code in RETRYABLE and body.get("errorCode") != "AuthUserDailyUsageExceeded"
        if not retryable or attempt == max_retries:
            raise WebIQError(response.status_code, body)
        retry_after = body.get("retryAfter")
        wait = int(str(retry_after).rstrip("s")) if retry_after else 2 ** attempt
        print(f"⏳ {response.status_code} {body.get('errorCode')} — retrying in {wait}s")
        time.sleep(wait)


print(f"✅ Ready — calling {WEBIQ_BASE_URL}")

---

## Step 1 — Web Search and the Response Anatomy

`POST /search/web` is the workhorse: public web content, compliantly retrieved and ranked for grounding.

| Parameter | Default | Notes |
|---|---|---|
| `query` | *required* | Up to 1,000 characters. Supports `site:` / `-site:` operators |
| `maxResults` | `10` | Up to **50** |
| `language` / `region` | `en` / `US` | ISO 639-1 language and 2-letter country/region code |
| `location` | inferred | `lat:<float>;long:<float>` for location-aware results |
| `contentFormat` | `html` | `passage`, `text`, `html`, or `markdown` — see Step 2 |
| `maxLength` | `10000` | Max characters per result `content` (up to 500,000) |
| `safeSearch` | `strict` | `strict` or `off` |

Besides `title`, `url`, and `content`, each result can carry **freshness and provenance signals** you can pass straight to your model or show in your UI: `crawledAt`, `lastUpdatedAt`, `language`, `isAdult`, and `contentTier` (`standard`, `entitled`, or `premium`). The response also includes query-level signals: `querySignals.freshness` (should fresh documents be preferred for this query?) and `riskLevel` (`low` → `veryHigh`).

In [ ]:
web = webiq_post("/search/web", {
    "query": "latest trends in retrieval-augmented generation",
    "maxResults": 5,
    "contentFormat": "passage",
    "maxLength": 1500,
})

print(f"querySignals: {web.get('querySignals')}   riskLevel: {web.get('riskLevel')}   traceId: {web['traceId']}\n")

for i, r in enumerate(web["webResults"], start=1):
    print(f"[{i}] {r['title']}")
    print(f"    {r['url']}")
    print(f"    crawledAt={r.get('crawledAt', '—')}  lastUpdatedAt={r.get('lastUpdatedAt', '—')}  "
          f"language={r.get('language')}  contentTier={r.get('contentTier', '—')}")
    print(f"    {r['content'][:220]}...\n")

> 💡 `crawledAt` and `lastUpdatedAt` aren't available for every page, so always read them with `.get()`. Keep the `traceId` — Web IQ Support asks for it when you report an issue.

---

## Step 2 — Content Formats and Token Efficiency

What you send to your model is what you pay for. `contentFormat` controls what comes back in `content`:

| Format | What you get | Best for |
|---|---|---|
| `passage` | Query-contextual extracted paragraphs, plain text | **LLM grounding** — the most token-efficient option |
| `text` | Full document, plain text | Summarizing or reasoning over a whole page |
| `markdown` | Full document, Markdown | Preserving headings, lists, and tables |
| `html` *(default for Web)* | Full document, HTML | Your own parsing or rendering |

Give each format the **same character budget** and compare what fills it:

In [ ]:
query = "how do AI agents use web grounding to reduce hallucinations"

for fmt in ["passage", "markdown", "html"]:
    r = webiq_post("/search/web", {
        "query": query,
        "maxResults": 1,
        "contentFormat": fmt,
        "maxLength": 500,
    })["webResults"][0]
    print(f"── {fmt} ({len(r['content'])} chars) — {r['title']}")
    print(r["content"][:500], "\n")

With the same budget, `passage` spends it on the paragraphs that actually answer the query, while the full-document formats spend it on the top of the page — titles, bylines, and markup. That's the "fewer tokens in, better answers out" idea behind Web IQ: every token you send to your model earns its place.

**`maxLength` is your token budget dial.** It caps `content` per result, so `maxResults × maxLength` bounds the context you'll send to your model (we approximate tokens as characters ÷ 4):

In [ ]:
print(f"{'maxLength':>9} {'results':>7} {'total chars':>11} {'~tokens':>8}")
print("-" * 38)
for max_length in [500, 2000, 10000]:
    resp = webiq_post("/search/web", {
        "query": query,
        "maxResults": 5,
        "contentFormat": "passage",
        "maxLength": max_length,
    })
    total = sum(len(r.get("content", "")) for r in resp["webResults"])
    print(f"{max_length:>9,} {len(resp['webResults']):>7} {total:>11,} {total // 4:>8,}")

---

## Step 3 — Targeting: `site:`, Language, Region, and Location

Web IQ covers 100+ languages and markets. Three levers help you scope results:

- **`site:` / `-site:`** in the query restrict to (or exclude) a domain. Use them sparingly — they reduce relevance ranking freedom.
- **`language` + `region`** set the market (for example `de` + `DE`). See [Supported languages and regions](https://webiq.microsoft.ai/documentation/supported-languages-and-regions/?view=md).
- **`location`** (`lat:<float>;long:<float>`) makes results location-aware. If omitted, location is inferred from the query.

In [ ]:
def show(resp: dict, key: str = "webResults", n: int = 3):
    for r in resp[key][:n]:
        print(f"  • {r['title']}\n    {r['url']}")
    print()

print("🔎 site: operator — only learn.microsoft.com")
show(webiq_post("/search/web", {"query": "agentic retrieval site:learn.microsoft.com", "maxResults": 3, "contentFormat": "passage"}))

print("🌍 language=de, region=DE")
show(webiq_post("/search/web", {"query": "Künstliche Intelligenz Agenten", "language": "de", "region": "DE", "maxResults": 3, "contentFormat": "passage"}))

print("📍 location = Seattle, WA")
show(webiq_post("/search/web", {"query": "coffee shops near me", "location": "lat:47.6062;long:-122.3321", "maxResults": 3, "contentFormat": "passage"}))

---

## Step 4 — News Search

`POST /search/news` returns trusted news coverage from the **last 14 days only** — use Web Search for anything older. Each result adds news-specific fields: `source` (the publisher), `snippet` (a terse search-style summary), an optional `thumbnail`, and `lastUpdatedAt`.

| Parameter | Default | Notes |
|---|---|---|
| `maxResults` | `10` | Up to **20** |
| `contentFormat` | `text` | `passage`, `text`, `html`, or `markdown` |
| `maxLength` | `10000` | Up to 500,000 |

In [ ]:
news = webiq_post("/search/news", {
    "query": "Microsoft AI announcements",
    "maxResults": 5,
    "contentFormat": "passage",
    "maxLength": 800,
})

for r in news["newsResults"]:
    print(f"📰 {r['title']}")
    print(f"   {r.get('source', 'unknown source')} · {r.get('lastUpdatedAt') or r.get('crawledAt', '')}")
    print(f"   {r['url']}")
    print(f"   {(r.get('snippet') or r.get('content', ''))[:200]}...\n")

---

## Step 5 — Images Search with Filters

`POST /search/images` returns images with a generated `caption`, the `hostPageUrl` the image lives on (cite this, not just the image file), dimensions, and a thumbnail. Filters let you shape results for your UI:

| Filter | Values |
|---|---|
| `aspectRatio` | `square`, `wide`, `tall` |
| `imageSize` | `small` (<200px), `medium` (200–500px), `large` (≥500px), `extraLarge` (>1024×768) |
| `color` | `colorOnly`, `monochrome` |
| `watermarkFree` | `true` to exclude watermarked images |
| `minWidth` / `maxWidth` / `minHeight` / `maxHeight` | Exact pixel bounds (take precedence over `imageSize`) |

In [ ]:
images = webiq_post("/search/images", {
    "query": "Seattle skyline at night",
    "maxResults": 3,
    "aspectRatio": "wide",
    "imageSize": "large",
    "watermarkFree": True,
})

for r in images["imageResults"]:
    print(f"🖼️  {r['title']}  ({r.get('width')}×{r.get('height')})")
    print(f"    caption:  {r.get('caption', '—')}")
    print(f"    image:    {r['url']}")
    print(f"    hostPage: {r.get('hostPageUrl', '—')}\n")

---

## Step 6 — Videos Search with Moments

`POST /search/videos` returns videos with publisher, view count, length, an AI-generated `summary`, and embedding info (`embeddingUrl`, `allowHttpsEmbedding`). Where available, `moments` break a video into segments with deep links, and `bestMomentIndex` points to the most relevant one — so your agent can cite *the exact part* of a video.

Useful filters: `freshness` (`week`, `month`, `year`, or an ISO 8601 range like `2026-01-01/..`), `duration` (`short` <5 min, `medium` 5–20 min, `long` >20 min), `resolution` (minimum `360p`–`1080p`), `embeddable: ["player"]`, and `enablePlaylist`. `maxResults` goes up to **30**.

In [ ]:
videos = webiq_post("/search/videos", {
    "query": "Microsoft Build keynote AI agents",
    "maxResults": 3,
    "freshness": "year",
    "resolution": "720p",
})

for r in videos["videoResults"]:
    print(f"🎬 {r['title']}")
    print(f"   {r.get('publishedBy', '—')} · {r.get('length', '—')} · {r.get('viewCount', 0):,} views")
    print(f"   {r['url']}")
    moments = r.get("moments") or []
    best = r.get("bestMomentIndex")
    if moments and best is not None and best < len(moments):
        m = moments[best]
        print(f"   ⭐ Best moment: {m.get('title')} ({m.get('startTime')}–{m.get('endTime')}) {m.get('momentUrl', '')}")
    print()

---

## Step 7 — Browse a Specific URL

Search finds pages; **Browse** reads one. `POST /browse` extracts grounding content from a URL you already have — a link the user pasted, a citation you want to expand, or a page your agent decided to open.

| Parameter | Default | Notes |
|---|---|---|
| `url` | *required* | The page to read |
| `contentFormat` | `html` | `text`, `html`, or `markdown` |
| `maxLength` | `10000` | Up to 500,000 |
| `liveCrawl` | `none` | `none` = index only (returns `404` if not indexed). `fallback` = crawl on demand if not indexed |
| `renderDynamicPages` | `false` | Run JavaScript before extracting (only with `liveCrawl="fallback"`) |
| `includeWebLinks` / `includeImageLinks` | `false` | Include outbound links / image URLs |

With `liveCrawl="fallback"`, an un-indexed page returns **`202 Accepted`** with a `retryAfter` (for example `"60s"`) while Web IQ crawls it. Wait, then retry the same request. The helper below handles that.

In [ ]:
def browse(url: str, max_polls: int = 5, **options) -> dict:
    payload = {"url": url, "contentFormat": "markdown", "maxLength": 1500, "liveCrawl": "fallback", **options}
    for _ in range(max_polls):
        response = session.post(f"{WEBIQ_BASE_URL}/browse", json=payload, timeout=60)
        if response.status_code in (202, 429):
            wait = int(str(response.json().get("retryAfter") or "10s").rstrip("s"))
            print(f"⏳ {response.status_code} — retrying in {wait}s")
            time.sleep(wait)
            continue
        if response.status_code == 200:
            return response.json()
        raise WebIQError(response.status_code, response.json())
    raise TimeoutError(f"Page was still being crawled after {max_polls} attempts")


page = browse("https://github.com/microsoft/iq-series")
print(f"📄 {page['title']}")
print(f"   crawledAt={page.get('crawledAt', '—')}  lastUpdatedAt={page.get('lastUpdatedAt', '—')}\n")
print(page["content"][:1000])

---

## Step 8 — Auto (Beta): Typed Evidence with Citations

`POST /auto` is a single entry point that decides which verticals to consult and returns **typed evidence**, each with its own **result-scoped citations**. Beyond web and news text, it can return structured answers like weather and time zones.

| Parameter | Default | Notes |
|---|---|---|
| `query` | *required* | Supports `site:` / `-site:` |
| `mode` | `fast` | `fast` prioritizes speed; `advanced` prioritizes comprehensiveness |
| `responseFilter` | all | Any of `web`, `news`, `timezone`, `weather`, `images`, `videos` |
| `maxResponseLength` | `100000` | Total response budget, 1,500–500,000 characters |
| `location`, `language`, `region` | — | Same as Web Search |

Evidence `type` tells you how to read `content`: `text` (web/news), `media/image`, `media/video`, `structured/weather`, `structured/timeZone`, `structured/sports`, `structured/places`.

> ⚠️ Auto is in **Beta** and is only available to profiles enabled for it. If your profile isn't enabled, the cell below reports that and you can move on.

In [ ]:
try:
    auto = webiq_post("/auto", {
        "query": "Seattle weather today and latest Microsoft AI news",
        "mode": "fast",
        "responseFilter": ["web", "news", "weather"],
        "location": "lat:47.6062;long:-122.3321",
        "maxResponseLength": 8000,
    })
    for item in auto["results"][:6]:
        evidence = item["evidence"]
        content = evidence["content"]
        preview = content[:160] if isinstance(content, str) else json.dumps(content)[:160]
        print(f"🧩 type={evidence['type']}")
        print(f"   {preview}...")
        for c in item.get("citations", [])[:2]:
            print(f"   ↳ [{c.get('contentSource')}] {c.get('title')} — {c.get('url')}")
        print()
except WebIQError as e:
    print(f"Auto (Beta) isn't available for this profile: {e}")

---

## Step 9 — Ground a Model's Answer with Citations

Now let's put it together. Web IQ is **model-agnostic** — it returns structured, citation-ready context and leaves generation to you. The pattern is:

1. **Retrieve** — call Web Search (`passage` format) and News Search for the user's question.
2. **Number the sources** — give each result an ID like `[1]`, `[2]` and keep its URL.
3. **Ground** — send the numbered sources plus the question to your model with instructions to answer *only* from the sources and cite them.
4. **Show citations** — print the answer followed by the source list, so users can verify every claim.

This step uses a model deployed in **Microsoft Foundry** if `FOUNDRY_PROJECT_ENDPOINT` is set. If it isn't, the cell prints the grounded prompt so you can paste it into any model.

In [ ]:
question = "What has Microsoft announced recently about AI agents?"

web_ctx = webiq_post("/search/web", {"query": question, "maxResults": 4, "contentFormat": "passage", "maxLength": 1200})
news_ctx = webiq_post("/search/news", {"query": question, "maxResults": 3, "contentFormat": "passage", "maxLength": 1200})

sources = []
for r in web_ctx["webResults"]:
    sources.append({"title": r["title"], "url": r["url"], "date": r.get("lastUpdatedAt") or r.get("crawledAt", ""), "content": r["content"]})
for r in news_ctx["newsResults"]:
    sources.append({"title": f"{r['title']} ({r.get('source', 'news')})", "url": r["url"], "date": r.get("lastUpdatedAt") or r.get("crawledAt", ""), "content": r.get("content", "")})

context = "\n\n".join(
    f"[{i}] {s['title']}\nURL: {s['url']}\nDate: {s['date']}\n{s['content']}"
    for i, s in enumerate(sources, start=1)
)

instructions = (
    "You are a helpful research assistant. Answer the user's question using ONLY the numbered sources provided. "
    "Cite every claim with the source number in square brackets, like [1] or [2][4]. "
    "If the sources don't contain the answer, say so. Keep the answer under 150 words."
)
grounded_input = f"Sources:\n\n{context}\n\nQuestion: {question}"

print(f"Retrieved {len(sources)} sources (~{len(grounded_input) // 4:,} tokens of context)")

In [ ]:
FOUNDRY_PROJECT_ENDPOINT = os.environ.get("FOUNDRY_PROJECT_ENDPOINT")
FOUNDRY_MODEL_DEPLOYMENT_NAME = os.environ.get("FOUNDRY_MODEL_DEPLOYMENT_NAME", "gpt-4o-mini")

if FOUNDRY_PROJECT_ENDPOINT:
    from azure.identity import DefaultAzureCredential
    from azure.ai.projects import AIProjectClient

    project = AIProjectClient(endpoint=FOUNDRY_PROJECT_ENDPOINT, credential=DefaultAzureCredential())
    openai_client = project.get_openai_client()
    response = openai_client.responses.create(
        model=FOUNDRY_MODEL_DEPLOYMENT_NAME,
        instructions=instructions,
        input=grounded_input,
    )
    print(f"🤖 {response.output_text}\n")
else:
    print("ℹ️ FOUNDRY_PROJECT_ENDPOINT not set — here's the grounded prompt to use with any model:\n")
    print(instructions + "\n\n" + grounded_input[:1500] + "...\n")

print("Sources:")
for i, s in enumerate(sources, start=1):
    print(f"  [{i}] {s['title']} — {s['url']}")

---

## Step 10 — Errors, Rate Limits, and Retries

Every Web IQ error uses the same JSON schema, so one handler covers every endpoint:

| Field | Description |
|---|---|
| `errorCode` | Machine-readable code, for example `HandlerInvalidInput`, `AuthInvalidApiKey`, `AuthUserThrottled` |
| `errorCategory` | `UserError` (fix the request) or `SystemError` (retry) |
| `userMessage` / `technicalDetails` | Human-readable message and debugging details |
| `retryAfter` | For `429` and retryable `5xx` — for example `"60s"` |
| `requestId` / `traceId` | Include these when contacting [Web IQ Support](https://aka.ms/microsoft-webiq-support) |

| Status | Meaning | What to do |
|---|---|---|
| `400` | Invalid parameters (for example `maxResults` over the limit) | Fix the request; validate locally |
| `401` / `403` | Bad or missing key / key lacks permission for this service | Check the key and allowed services in Profile Management |
| `404` *(Browse)* | URL not indexed with `liveCrawl="none"` | Retry with `liveCrawl="fallback"` |
| `202` *(Browse)* | Live crawl in progress — not an error | Wait `retryAfter`, then retry |
| `410` / `415` | HTTP instead of HTTPS / wrong `content-type` | Use HTTPS and `application/json` |
| `429` | `AuthUserThrottled` (per-second/minute quota) or `AuthUserDailyUsageExceeded` (daily cap, resets 00:00 UTC) | Back off and honor `retryAfter` |
| `430` *(Browse)* | Too many on-demand crawls | Slow down live crawl requests |
| `500` / `503` / `504` | Transient server, outage, or timeout | Retry with exponential backoff |

Let's trigger two errors on purpose to see the schema:

In [ ]:
for label, headers, payload in [
    ("400 — maxResults above the limit", {}, {"query": "test", "maxResults": 1000}),
    ("401 — invalid API key", {"x-apikey": "not-a-real-key"}, {"query": "test"}),
]:
    r = session.post(f"{WEBIQ_BASE_URL}/search/web", json=payload, headers=headers, timeout=30)
    body = r.json()
    print(f"{label}\n  status={r.status_code} errorCode={body.get('errorCode')} "
          f"category={body.get('errorCategory')}\n  {body.get('userMessage')} | {body.get('technicalDetails')}\n")

The `webiq_post` helper you defined in **Setup** already follows the recommended retry policy:

- Retry **only** transient statuses — `429`, `500`, `503`, `504`.
- Honor `retryAfter` when the service sends it; otherwise back off exponentially (1s, 2s, 4s, …).
- Don't retry `AuthUserDailyUsageExceeded` — the daily cap won't reset until 00:00 UTC.
- Never retry `4xx` user errors — fix the request instead.

Per-minute quotas (for example `60 QPM`) are easy to hit when you re-run a notebook back-to-back, so keep retries in every production client. Browse's `202` polling (Step 7) is separate: it's a normal "come back later" response, not an error.

---

## Step 11 — Web IQ as an MCP Server

Web IQ is **MCP-native**. The same capabilities are exposed as tools on a Streamable HTTP MCP server at `https://api.microsoft.ai/v3/mcp`, authenticated with the same `x-apikey` header (or Entra ID bearer token). Public tools include `web`, `news`, `images`, `videos`, and `browse`, and each accepts the same parameters as its REST counterpart. Your profile may expose additional tools.

### Use it from VS Code / GitHub Copilot

Add this to `.vscode/mcp.json`. VS Code prompts for the key once and stores it securely, so it never lands in source control:

```json
{
  "inputs": [
    { "type": "promptString", "id": "webiq-api-key", "description": "Web IQ API key", "password": true }
  ],
  "servers": {
    "web-iq": {
      "type": "http",
      "url": "https://api.microsoft.ai/v3/mcp",
      "headers": { "x-apikey": "${input:webiq-api-key}" }
    }
  }
}
```

Start the server, enable its tools in Copilot Chat's tool picker, and ask *"What's the latest news about Microsoft IQ? Cite your sources."*

### Call it directly with JSON-RPC 2.0

Any agent framework that speaks MCP can use Web IQ. Under the hood it's plain JSON-RPC — `tools/list` to discover tools, `tools/call` to invoke one. Responses wrap the same data as REST in `structuredContent`:

In [ ]:
MCP_URL = f"{WEBIQ_BASE_URL}/mcp"
MCP_HEADERS = {"accept": "application/json, text/event-stream"}


def mcp(method: str, params: dict | None = None, request_id: int = 1) -> dict:
    body = {"jsonrpc": "2.0", "method": method, "id": request_id}
    if params:
        body["params"] = params
    r = session.post(MCP_URL, json=body, headers=MCP_HEADERS, timeout=60)
    r.raise_for_status()
    return r.json()["result"]


tools = mcp("tools/list")["tools"]
print("🧰 Tools on this profile:", ", ".join(t["name"] for t in tools))

result = mcp("tools/call", {"name": "web", "arguments": {"query": "Microsoft IQ", "maxResults": 2}}, request_id=2)
for r in result["structuredContent"]["webResults"]:
    print(f"  • {r['title']} — {r['url']}")

---

## Step 12 — Instrumentation: Citation and Click Telemetry

Web IQ can measure **which results your model actually cited** and **which citations users clicked**, so you can understand grounding quality in production. Instrumentation is enabled per profile — ask your Microsoft contact to turn it on.

When it's enabled, responses include:

| Field | Purpose |
|---|---|
| `instrumentationCitationBase` | Base URL for **LLM citation** pings — append a URL-encoded JSON payload |
| `instrumentationClickBase` | Base URL for **user click** pings — append a result's `instrumentationSuffix` |
| `instrumentationSuffix` *(per result)* | Identifier such as `DevEx,1000` (`AppNS,K`) |
| `clickUrl` *(per result)* | Redirect URL that logs clicks automatically — when present, link users to it instead of `url` and no click ping is needed |

The workflow: (1) show citations in the model's answer, (2) send one citation ping listing the results the model used, and (3) send a click ping when a user opens a citation (or just link to `clickUrl`).

In [ ]:
def citation_ping_url(response: dict, cited_results: list[dict]) -> str | None:
    base = response.get("instrumentationCitationBase")
    if not base:
        return None
    events = []
    for r in cited_results:
        app_ns, k = r["instrumentationSuffix"].split(",", 1)
        events.append({"T": "CI.Info", "FID": "LLM", "Name": "citation", "AppNS": app_ns, "K": k})
    return base + urllib.parse.quote(json.dumps(events, separators=(",", ":")))


cited = web_ctx["webResults"][:2]  # e.g. the results the model cited as [1] and [2]
ping = citation_ping_url(web_ctx, cited)

if ping:
    print(f"📈 Citation ping URL (send with GET):\n{ping}")
elif any("clickUrl" in r for r in cited):
    print("ℹ️ Citation pings aren't enabled on this profile, but click redirect is — link users to clickUrl:")
    for r in cited:
        print(f"  • {r['title']}\n    {r['clickUrl'][:90]}...")
else:
    print("ℹ️ Instrumentation isn't enabled for this profile. Contact your Microsoft contact to enable it.")

---

## Conclusion

You've now used every part of Web IQ:

1. **Web Search** — ranked, citation-ready pages with freshness and provenance signals.
2. **Content formats** — `passage` fills your budget with query-relevant text, and `maxLength` sets that budget.
3. **Targeting** — `site:`, language/region markets, and location-aware results.
4. **News, Images, and Videos** — vertical-specific results, filters, and video moments.
5. **Browse** — read a specific URL, with on-demand live crawl and `202` polling.
6. **Auto (Beta)** — typed evidence with result-scoped citations in one call.
7. **Grounded answers** — numbered sources in, cited answers out, with any model.
8. **Errors and retries** — one error schema, retry only what's transient.
9. **MCP** — the same capabilities as tools for VS Code, GitHub Copilot, and any MCP client.
10. **Instrumentation** — measure what your model cites and what users click.

### Web IQ vs. Grounding with Bing

**Grounding with Bing** is designed for integrated web augmentation inside Microsoft Foundry Agent Service and remains available as an accessible entry point. **Web IQ** is purpose-built for agents and multi-step workflows: it returns structured, passage-level content **directly to you**, giving you full control over retrieval, orchestration, and how results are used in your LLM workflow.

### Where to go next

- **Combine with Foundry IQ** — Ground the same agent in your organization's knowledge (Foundry IQ) *and* the live web (Web IQ).
- **Experiment in the [Playground](https://webiq.microsoft.ai/playground/)** — Try parameters interactively and export code in seven languages.
- **Read the docs** — [Quick Start](https://webiq.microsoft.ai/documentation/quick-start/?view=md) · [API reference (OpenAPI)](https://webiq.microsoft.ai/documentation/openapi.json) · [MCP Server](https://webiq.microsoft.ai/documentation/mcp/?view=md) · [Error Handling](https://webiq.microsoft.ai/documentation/error-handling/?view=md) · [Instrumentation](https://webiq.microsoft.ai/documentation/instrumentation/?view=md)
- **Explore the IQ Series** — Head back to the [Web IQ overview](../../README.md) or continue with [Foundry IQ](../../../Foundry-IQ/) and [Work IQ](../../../Work-IQ/).